In [8]:
from faker import Faker
fake = Faker('en_GB')
Faker.seed(42)

In [13]:
n_sensors = 50
n_readings = 5000

In [4]:
from unidecode import unidecode
import uuid

In [5]:
from datetime import datetime, timedelta

In [25]:
from faker.providers import DynamicProvider

sensor = {}
sensor_type = ['temperature', 'humidity', 'air_quality']

unique_districts = set()
while len(unique_districts) < 15:
    unique_districts.add(fake.administrative_unit())

# 2. Register these 15 districts into a DynamicProvider to override the default method
district_provider = DynamicProvider(
     provider_name="administrative_unit",
     elements=list(unique_districts),
)
fake.add_provider(district_provider)

for i in range(n_sensors):
    latitude = fake.latitude()
    longitude = fake.longitude()
    sensor[i] = {
        'sensor_id': str(uuid.uuid4()),
        'district': fake.administrative_unit(),
        'sensor_type': fake.random_element(elements=sensor_type),
        'latitude_longitude': f"{latitude}, {longitude}",
        'installed_at': fake.date_time()
    }

print(sensor[0])

{'sensor_id': 'd6e8ccd6-355c-4281-baa5-c92a5d7f6355', 'district': 'Perth and Kinross', 'sensor_type': 'humidity', 'latitude_longitude': '-45.6831405, -53.723792', 'installed_at': datetime.datetime(2000, 9, 4, 17, 54, 13, 202459)}


In [17]:
sensor_id = [sensor[i]['sensor_id'] for i in range(n_sensors)]

In [ ]:
readings = {}

for i in range(n_readings):
    readings[i] = {
        'reading_id': str(uuid.uuid4()),
        'sensor_id': sensor_id[i % n_sensors],
        'value': round(fake.random.gauss(mu=20, sigma=5), 2) if sensor[i % n_sensors]['sensor_type'] == 'temperature' else (round(fake.random.gauss(mu=50, sigma=10), 2) if sensor[i % n_sensors]['sensor_type'] == 'humidity' else round(fake.random.gauss(mu=100, sigma=20), 2)),
        'unit': '°C' if sensor[i % n_sensors]['sensor_type'] == 'temperature' else ('%' if sensor[i % n_sensors]['sensor_type'] == 'humidity' else 'AQI'),
        'recorded_at': fake.date_time_between(start_date=sensor[i % n_sensors]['installed_at'], end_date=datetime.now()),
        'is_anomaly': fake.boolean(chance_of_getting_true=7)
    }

print(readings[0])

{'reading_id': '131c85e7-e6e0-4988-a38b-22029841a139', 'sensor_id': 'e56f6a52-84cd-4c8e-934e-efcc29b132a8', 'value': 14.0, 'unit': '°C', 'recorded_at': datetime.datetime(2016, 5, 31, 18, 48, 58, 486572), 'is_anomaly': False}


In [22]:
import pandas as pd

In [26]:
sensors_df = pd.DataFrame.from_dict(sensor, orient='index')
readings_df = pd.DataFrame.from_dict(readings, orient='index')

In [27]:
sensors_df.to_csv('sensors_raw.csv', index=False)
readings_df.to_csv('readings_raw.csv', index=False)